In [ ]:
!pip install "rembg[cpu]" -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from rembg import remove
from PIL import Image
from pathlib import Path
import os

SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff"}
src_root = Path("/content/drive/MyDrive/Gh0st in the Loop/dataset/raw/tricky")
bg_removed_root = Path("/content/drive/MyDrive/Gh0st in the Loop/dataset/bg_removed")

# Sets identified as having background issues
problem_sets = [
    "nat_indian",
    "smiley_holo",
    "camo_studio",
]

for set_name in problem_sets:
    input_dir = src_root / set_name
    output_dir = bg_removed_root / set_name
    # print(f"{input_dir=}")
    # print(f"{output_dir=}")
    output_dir.mkdir(parents=True, exist_ok=True)

    images = list(input_dir.rglob("*"))
    images = [p for p in images if p.suffix.lower() in SUPPORTED_EXTENSIONS]
    print(f"\n{set_name} — {len(images)} images")

    for i, path in enumerate(images, 1):
        out_path = output_dir / path.name
        if out_path.exists():
            print(f"  [{i}/{len(images)}] {path.name} ... skipped")
            continue

        try:
            img = Image.open(path).convert("RGBA")
            result = remove(img)
            # Paste on black background
            bg = Image.new("RGB", img.size, (0, 0, 0))
            bg.paste(result.convert("RGB"), mask=result.split()[3])
            bg.save(out_path)
            print(f"  [{i}/{len(images)}] {path.name} ... ok")
        except Exception as e:
            print(f"  [{i}/{len(images)}] {path.name} ... ERROR: {e}")

print("\nDone. Review results in bg_removed/ before overwriting prepared/.")